This notebook runs the distinguishing trace generation experiments in the evaluation

You can select the benchmark and structured NL in the next two cells

Then you may run all remaining cells to run distinguishing trace generation with LTLTalk and ARTEMIS

Results can be viewed in Plot_results.ipynb

To convenientialy run all cells: Select Run -> Run all cells option from the top of this notebook to view the plots

In [ ]:
import os
os.environ["STRUCTNL_MODE"] = "fretish" #or "PSP"

In [ ]:
#select the benchmark

#cur_dataset_name = "Ventilator"
cur_dataset_name = "FSM-AP"
#cur_dataset_name = "FSM-S"
#cur_dataset_name = "REG"
#cur_dataset_name = "RobotExplain"
#cur_dataset_name = "deepstl-test"
#cur_dataset_name = "Thales"

In [ ]:
os.environ["DATA_HOME_DIR"] = "./metadata"
os.environ["SMV_FILE_DIR"] = "./metadata/tmp"

In [ ]:
import metrics
import data_loader
import batch_check
import spot_utils
import dist_trace
import nl2structnl

In [ ]:
import numpy as np
import pandas as pd
import json
import itertools
import time

In [ ]:
if os.getenv("STRUCTNL_MODE") == "fretish":
    result_dir = "./fretish_results/"
    data_home_dir = "./benchmarks/fret_specs/"
else:
    result_dir = "./PSP_results/"
    data_home_dir = "./benchmarks/psp_specs/"

In [ ]:
cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
df = pd.read_excel(cur_df_file, engine='openpyxl')
len(df)

In [ ]:
num_trial = 50
model="gemini-2.5-flash"

In [ ]:
input_methods = ["nl2structnl-reflect"]
modes = [None]

In [ ]:
row_idx_range = [i for i in range(len(df))]

In [ ]:
if os.getenv("STRUCTNL_MODE") == "fretish":
    #constraint_aut = spot.automaton(os.getenv("DATA_HOME_DIR")+'/fretish_proxy_constraints.aut')
    #constraint_f_list = [f"({spot_utils.trace_to_formula(spot.product(constraint_aut,spot.translate('X F (!c1 & c2)')),debug=False)}) | ({spot_utils.trace_to_formula(spot.product(constraint_aut,spot.translate('X G (m1)')),debug=False)})"]
    #constraint_f_list = ["(((!(c1)) & (!(c2)) & (m3) & (!(m4)) & (m5) & (m6) & (!(m7)) & (X((c1) & (c2) & (m1) & (!(m2)) & (!(m3)) & (!(m4)) & (!(m5)) & (m6) & (m7) & (X(((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6)) U (((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))) | (G((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))))))) | ((!(c1)) & (!(c2)) & (m3) & (!(m4)) & (m5) & (!(m6)) & (!(m7)) & (X(((c1) & (c2) & (m1) & (!(m2)) & (!(m3)) & (!(m4)) & (!(m5)) & (m6) & (m7) & (X(((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6)) U (((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))) | (G((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))))) | ((c1) & (c2) & (m1) & (!(m2)) & (!(m3)) & (!(m4)) & (!(m5)) & (!(m6)) & (m7) & (X(((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (!(m6))) U (((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))) | ((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6)) U (((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))) | (G((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))))) | ((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (!(m6)) & (X(((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (!(m6))) U (((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6) & (X(G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (m6))))) | (G((!(c1)) & (!(c2)) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (!(m6)))))))) | (G((!(c1)) & (c2) & (m1) & (!(m2)) & (!(m4)) & (!(m5)) & (!(m6))))))))))))"]
    constraint_f_list = [f"({dist_trace.get_fretish_witness_trace()})"]
    proxy_start_step = 1 #the timestep the proxy semantics start, prior timesteps can be ignored
    dist_trace.proxy_start_step = proxy_start_step
    #exclude_vars = set([var.to_str() for var in constraint_aut.ap()])
    exclude_vars = set([var for var in spot_utils.get_variables_from_formula(constraint_f_list[0])])
elif os.getenv("STRUCTNL_MODE") == "PSP":
    proxy_start_step = 0
    dist_trace.proxy_start_step = proxy_start_step
    constraint_f_list = [f"({dist_trace.get_PSP_witness_trace()})"]
    exclude_vars = set([var for var in spot_utils.get_variables_from_formula(constraint_f_list[0])])
else:
    assert False

In [ ]:
dist_methods = ["ltltalk","abstract"]

In [ ]:
def load_metrics(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_mode,cur_method):
    cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
    if cur_mode == "extra":
        save_name = cur_mode + "_" + cur_method
    else:
        save_name = cur_method
    with open(cur_exp_name + "_" + save_name + "_metrics.json", "r") as json_file:
        pred_metrics = json.load(json_file)
    return pred_metrics

def is_need_run(in_file, res_file):
    assert os.path.exists(in_file)
    if not os.path.exists(res_file):
        return True  # No result yet
    mtime_input = os.path.getmtime(in_file)
    mtime_result = os.path.getmtime(res_file)
    return mtime_input >= mtime_result

def get_max_num_trial_outputs(result_dir,cur_dataset_name,row_idx,model,cur_method,cur_mode,max_N_DURATION=None):
    for num_trial in [50*(i+1) for i in range(20)][::-1]:
        try:
            cur_outputs, output_ltl_list = data_loader.load_outputs(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,structnl=os.getenv("STRUCTNL_MODE"))
            if num_trial != len(output_ltl_list):
                print(row_idx,len(output_ltl_list),num_trial)
            break
        except:
            pass
    return cur_outputs, output_ltl_list

def get_num_candidate_to_first_plausible(output_metrics):
    if np.sum(output_metrics,axis=0)[0] == 0:
        return len(output_metrics)
    else:
        return np.argmax(np.array(output_metrics)[:,0]) + 1

def prepare_dist_trace(result_dir, model, num_trial, cur_method, cur_mode, data_home_dir, cur_dataset_name, row_idx, max_N_DURATION):
    cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
    if cur_mode == "extra":
        save_name = cur_mode + "_" + cur_method
    else:
        save_name = cur_method
    if False and is_need_run(in_file=cur_exp_name+"_" + save_name + "_metrics.json",
                           res_file=cur_exp_name+"_" + save_name + "_out-cache.json"):
        label_output_list,label_ltl_list = data_loader.load_labels(data_home_dir,cur_dataset_name,row_idx,max_N_DURATION,structnl=os.getenv("STRUCTNL_MODE"))
        #cur_outputs, output_ltl_list = data_loader.load_outputs(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,structnl=os.getenv("STRUCTNL_MODE"))
        cur_outputs, output_ltl_list = get_max_num_trial_outputs(result_dir,cur_dataset_name,row_idx,model,cur_method,cur_mode)
        pred_metrics = load_metrics(result_dir,cur_dataset_name,row_idx,model,len(cur_outputs),cur_mode,cur_method)
        num_outputs = get_num_candidate_to_first_plausible(pred_metrics)
        cur_outputs,output_ltl_list = cur_outputs[:num_outputs],output_ltl_list[:num_outputs]
        
        total_ltl_list = output_ltl_list + label_ltl_list
        total_output_list = cur_outputs + label_output_list
    
        is_correct_pred = np.array(pred_metrics)[:,0].tolist() + [True for entry in label_ltl_list]

        unique_idx_list = batch_check.remove_equivalent_idx_old(total_ltl_list,mc_mode="timeout",bmc_k=20)

        target_idx_list = [idx for idx in unique_idx_list if is_correct_pred[idx]]
        
        all_ltl_list = [[total_ltl_list[idx]] for idx in unique_idx_list]
        all_target_ltl_list = [[total_ltl_list[idx]] for idx in target_idx_list]
                               
        new_output_list = [total_output_list[idx] for idx in unique_idx_list]
        target_output_list = [total_output_list[idx] for idx in target_idx_list]
        assert len(new_output_list) == len(all_ltl_list)
        assert len(all_target_ltl_list) == len(target_output_list)
    
        #configure dcmp
        """
        cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
        df = pd.read_excel(cur_df_file, engine='openpyxl')
        
        dcmp, dcmp_list = data_loader.get_dcmp_map_from_row(df.iloc[row_idx])
        
        for entry in new_output_list:
            for key,value in dcmp.items():
                entry[key] = value
        """
        dcmp, dcmp_list = None,None
        #get all variables
        total_var_dict = {}
        for f in all_ltl_list:
            total_var_dict.update(dict( (k,"boolean") for k in spot_utils.get_variables_from_formula(f[0])))
        total_var_dict["bool_exp3"] = "boolean"
        for var in exclude_vars:
            total_var_dict[var] = "boolean"
        with open(cur_exp_name+"_" + save_name + "_out-cache.json", "w") as json_file:
            save_data = (all_ltl_list, all_target_ltl_list, new_output_list, target_output_list, dcmp_list, total_var_dict)
            json.dump(save_data, json_file)
    else:
        with open(cur_exp_name+"_" + save_name + "_out-cache.json", "r") as json_file:
            all_ltl_list, all_target_ltl_list, new_output_list, target_output_list, dcmp_list, total_var_dict = json.load(json_file)
    return all_ltl_list, all_target_ltl_list, new_output_list, target_output_list, dcmp_list, total_var_dict

In [ ]:
def run_task(dist_methods,data_home_dir,result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,max_N_DURATION):
    mc_mode = "timeout"
    bmc_k=20
    try:
        all_ltl_list, all_target_ltl_list, new_output_list, target_output_list, dcmp_list, total_var_dict = prepare_dist_trace(result_dir,
                                                                                                        model,
                                                                                                        num_trial,
                                                                                                        cur_method,
                                                                                                        cur_mode,
                                                                                                        data_home_dir,
                                                                                                        cur_dataset_name,
                                                                                                        row_idx,
                                                                                                        max_N_DURATION)
        
        
        if os.getenv("STRUCTNL_MODE") == "fretish":
            dcmp = {
                "decision1_substring":"scope",
                "decision2_substring":"condition",
                "decision3_substring":"timing",
            }
            dcmp_list = ["scope","condition","timing"]
        elif os.getenv("STRUCTNL_MODE") == "PSP":
            dcmp = {
                "decision1_substring":"scope",
                "decision2_substring":"pattern",
            }
            dcmp_list = ["scope","pattern"]
        else:
            assert False
        for output in new_output_list:
            for k,v in dcmp.items():
                output[k] = v
        assert len(all_ltl_list) > 0
        assert len(all_target_ltl_list) > 0
        all_target_ltl_list = all_target_ltl_list[:1]
        target_output_list = target_output_list[:1]
        print("num candidates",len(all_ltl_list))
    except Exception as e:
        print(e)
        print("fail",cur_dataset_name,row_idx,model,cur_method)
        return
    
    print("num targ:",len(target_output_list),"num outputs:",len(all_ltl_list))
    if len(all_ltl_list) == 1:
        print("WARNING: only one unique spec, no distinguishing traces needed!")
        return
    for dist_method in dist_methods:
        cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
        if cur_mode == "extra":
            save_name = cur_mode + "_" + cur_method + "_" + dist_method
            out_save_name = cur_mode + "_" + cur_method
        else:
            out_save_name = cur_method
            save_name = cur_method + "_" + dist_method
        if True or is_need_run(in_file=cur_exp_name+"_" + out_save_name + "_metrics.json", res_file=cur_exp_name+f"_{save_name}"+"_traces.json"):
            print(dist_method)
            if dist_method == "balance":
                dist_trace.min_group_size = 10
            else:
                #dist_trace.min_group_size = None
                dist_trace.min_group_size = 10
            
            if dist_method in ["balance","bottomup","abstract"]:
                get_mask_func = dist_trace.get_maxmin_elim_mask
            elif dist_method in ["ltltalk"]:
                get_mask_func = dist_trace.get_myltltalk_elim_mask
            else:
                assert False
            
            if dist_method in ["ltltalk","balance"]:
                run_sim_func = dist_trace.get_query_log
            elif dist_method == "bottomup":
                run_sim_func = dist_trace.get_bottomup_distinguish
            elif dist_method == "abstract":
                run_sim_func = dist_trace.get_abstract_distinguish
            else:
                assert False
            
            count_results = []
            var_count = []
            tracesize = []
            balanced = []
            trace_gen_time = []
            if dist_method in ["ltltalk","balance"]:
                aut_cache = dist_trace.AutomatonCache()
                for target_f in all_target_ltl_list:
                    log,res_f = run_sim_func(target_f,total_var_dict,all_ltl_list,
                                                            get_mask_func=get_mask_func,
                                                            mc_mode=mc_mode,bmc_k=bmc_k,aut_cache=aut_cache)
                    log = [list(e)+["full",set(total_var_dict.keys())] for e in log]
                    assert target_f == res_f[0]
                    cur_num_traces, cur_var_count, cur_tracesize, cur_balanced, cur_gen_time = dist_trace.get_results_from_trace_log(log,mc_mode=mc_mode,bmc_k=bmc_k)
                    count_results.append(cur_num_traces)
                    var_count.append(cur_var_count)
                    tracesize.append(cur_tracesize)
                    balanced.append(cur_balanced)
                    trace_gen_time.append(cur_gen_time)
            elif dist_method in ["bottomup","abstract"]:
                aut_cache = dist_trace.AutomatonCache()
                for output in target_output_list:
                    target_options = nl2structnl.get_output_options(output)
                    log = run_sim_func(target_options,
                                                            new_output_list,
                                                            dcmp_list,
                                                            total_var_dict,
                                                            get_mask_func=get_mask_func,
                                                            mc_mode=mc_mode,bmc_k=bmc_k,
                                                            aut_cache=aut_cache,
                                                            constraint_f_list=constraint_f_list,
                                                           )
                    cur_num_traces, cur_var_count, cur_tracesize, cur_balanced, cur_gen_time = dist_trace.get_results_from_trace_log(log,mc_mode=mc_mode,exclude_vars=exclude_vars,proxy_start_step=proxy_start_step,bmc_k=bmc_k)
                    count_results.append(cur_num_traces)
                    var_count.append(cur_var_count)
                    tracesize.append(cur_tracesize)
                    balanced.append(cur_balanced)
                    trace_gen_time.append(cur_gen_time)
            cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
            if cur_mode == "extra":
                save_name = cur_mode + "_" + cur_method + "_" + dist_method
            else:
                save_name = cur_method + "_" + dist_method
            with open(cur_exp_name+f"_{save_name}"+"_traces.json", "w") as json_file:
                json.dump((count_results,var_count,tracesize,balanced,trace_gen_time), json_file)
            print("num traces:",np.sum(count_results))
            print("avg variables per trace:",np.mean(dist_trace.flatten_list(var_count)))
            #print(np.mean(dist_trace.flatten_list(tracesize)))
            b_arr = np.array(dist_trace.flatten_list(balanced))
            #print(np.mean(b_arr[:,0] >= b_arr[:,1]//2))
            #return count_results,var_count,tracesize,balanced

the following cell runs distinguishing trace generation evaluation, the number of traces and average number of variables per trace are printed at the end, but you can also view the results in Plot_results.ipynb

In [ ]:
for row_idx in row_idx_range:
    print("row_idx",row_idx)
    time.sleep(0.01)
    for cur_method, cur_mode in list(itertools.product(input_methods,modes)):
        run_task(dist_methods,data_home_dir,result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,max_N_DURATION=1)
    break #remove this line to run on the entire benchmark